In [25]:
# pip install plotly 
# Run this line once to install plotly if it's not already installed, then restart the kernel

In [26]:
import findspark
import logging

# Configure logging to show timestamp, module, level and message
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info("Initializing findspark")
# initialize findspark so Spark libraries can be imported in this notebook
findspark.init()
logger.info("Findspark initialized successfully")

2026-02-27 20:46:39,909 - __main__ - INFO - Initializing findspark
2026-02-27 20:46:39,923 - __main__ - INFO - Findspark initialized successfully


In [27]:
from pyspark.sql import SparkSession

logger.info("Creating SparkSession")
# initialize a SparkSession; if one already exists it will be returned
spark = SparkSession.builder \
    .appName("pyspark-1") \
    .getOrCreate()
logger.info(f"SparkSession created successfully: version {spark.version}")
# SparkSession provides the entry point for DataFrame and SQL functionality


2026-02-27 20:46:39,991 - __main__ - INFO - Creating SparkSession
2026-02-27 20:46:40,012 - __main__ - INFO - SparkSession created successfully: version 2.4.5


# Functions Import

In [28]:
import logging
import json
import os
import shutil

from pyspark.sql.functions import * 
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import *


from user_functions import *
from pre_process_functions import *

### Functions Definations

Preprocess Functions are present in pre_process_functions.py <br>
User Defined Functions are present in user_functions.py

In [29]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

# define a schema for the NYC Jobs CSV file to ensure proper types instead of inferring
# The schema is created based on the values of the column , in case there are any column with str values in int column then we create that col as string only.
# It can be then converted or transfrmed based on the usecase

jobs_schema = StructType([
    StructField("Job ID", IntegerType(), True),
    StructField("Agency", StringType(), True),
    StructField("Posting Type", StringType(), True),
    StructField("# Of Positions", IntegerType(), True),
    StructField("Business Title", StringType(), True),
    StructField("Civil Service Title", StringType(), True),
    StructField("Title Code No", StringType(), True),
    StructField("Level", StringType(), True),
    StructField("Job Category", StringType(), True),
    StructField("Full-Time/Part-Time indicator", StringType(), True),
    StructField("Salary Range From", DoubleType(), True),
    StructField("Salary Range To", DoubleType(), True),
    StructField("Salary Frequency", StringType(), True),
    StructField("Work Location", StringType(), True),
    StructField("Division/Work Unit", StringType(), True),
    StructField("Job Description", StringType(), True),
    StructField("Minimum Qual Requirements", StringType(), True),
    StructField("Preferred Skills", StringType(), True),
    StructField("Additional Information", StringType(), True),
    StructField("To Apply", StringType(), True),
    StructField("Hours/Shift", StringType(), True),
    StructField("Work Location 1", StringType(), True),
    StructField("Recruitment Contact", StringType(), True),
    StructField("Residency Requirement", StringType(), True),
    StructField("Posting Date", StringType(), True),
    StructField("Post Until", StringType(), True),
    StructField("Posting Updated", StringType(), True),
    StructField("Process Date", StringType(), True)
])


## Reading Data

In [30]:
logger.info("Reading CSV file: /dataset/nyc-jobs.csv")

# Try to load file using predefined schema & Throws errors if incorect
try:
    df = spark.read.csv('/dataset/nyc-jobs.csv', header=True, schema=jobs_schema)
except Exception as e:
    logger.error(f"Failed to load CSV: {e}")
    raise

logger.info("CSV file read into DataFrame")

total_rows = df.count()
total_cols = len(df.columns)
logger.info(f"DataFrame dimensions: {total_rows} rows, {total_cols} columns")

# print schema and counts
logger.info("Inspecting DataFrame schema:")
df.printSchema()
print("================================================================")
print(f"Total Rows : {total_rows}")
print(f"Total Columns : {total_cols}")

2026-02-27 20:46:40,316 - pre_process_functions - INFO - Reading CSV file: /dataset/nyc-jobs.csv
2026-02-27 20:46:40,447 - pre_process_functions - INFO - CSV file read into DataFrame
2026-02-27 20:46:40,682 - pre_process_functions - INFO - DataFrame dimensions: 2946 rows, 28 columns
2026-02-27 20:46:40,685 - pre_process_functions - INFO - Inspecting DataFrame schema:


root
 |-- Job ID: integer (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: integer (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: double (nullable = true)
 |-- Salary Range To: double (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locat

In [31]:
logger.info("Displaying the DataFrame")
display(df)

2026-02-27 20:46:40,734 - pre_process_functions - INFO - Displaying the DataFrame


+------+------------------------------+------------+--------------+-------------------------------------------------------------------------------------+------------------------------+-------------+-----+------------------------------------------------------------------------------+-----------------------------+-----------------+---------------+----------------+------------------------------+------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Data Analysis

In [32]:
import pandas as pd

logger.info("Initiating exploratory analysis on the dataset")
logger.info("Reading raw CSV into pandas for quick inspection")
raw = pd.read_csv('/dataset/nyc-jobs.csv')
logger.info(f"Raw data dimensions: {raw.shape[0]} rows, {raw.shape[1]} columns")

# summarise each column: data type, counts, nulls, uniques
logger.info("Analysis on Column Data Types , Counts , Nulls & Unqiues")
column_summary = pd.DataFrame({
    'column': raw.columns,
    'dtype': raw.dtypes.astype(str),
    'non_null': raw.count(),
    'nulls': raw.isna().sum(),
    'null_pct': raw.isna().mean() * 100,
    'unique': raw.nunique()
})

logger.info("Completed column summary")

# collect categories with limited cardinality for value counts
logger.info("Identifying low-cardinality object columns ie Categorical Columns")
object_cols = raw.select_dtypes(include=['object']).columns
object_cols = [c for c in object_cols if 'date' not in c.lower()]
category_counts = {}
for _col in object_cols:
    if raw[_col].nunique() <= 20:
        category_counts[_col] = raw[_col].value_counts().to_dict()
logger.info(f"Categorical columns analysed: {len(category_counts)}")

# statistics for numeric fields
logger.info("Generating descriptive statistics for numeric features")
numeric_stats = raw.describe()

logger.info("Exploratory data preparation finished")

print("Column Summary:\n", column_summary.to_string())
print("\nCategorical Value Counts:\n", category_counts)
print("\nNumeric Descriptive Stats:\n", numeric_stats.to_string())

2026-02-27 20:46:41,250 - pre_process_functions - INFO - Initiating exploratory analysis on the dataset
2026-02-27 20:46:41,253 - pre_process_functions - INFO - Reading raw CSV into pandas for quick inspection
2026-02-27 20:46:42,207 - pre_process_functions - INFO - Raw data dimensions: 2946 rows, 28 columns
2026-02-27 20:46:42,209 - pre_process_functions - INFO - Analysis on Column Data Types , Counts , Nulls & Unqiues
2026-02-27 20:46:42,412 - pre_process_functions - INFO - Completed column summary
2026-02-27 20:46:42,413 - pre_process_functions - INFO - Identifying low-cardinality object columns ie Categorical Columns
2026-02-27 20:46:42,472 - pre_process_functions - INFO - Categorical columns analysed: 4
2026-02-27 20:46:42,474 - pre_process_functions - INFO - Generating descriptive statistics for numeric features
2026-02-27 20:46:42,517 - pre_process_functions - INFO - Exploratory data preparation finished


Column Summary:
                                                       column    dtype  non_null  nulls    null_pct  unique
Job ID                                                Job ID    int64      2946      0    0.000000    1661
Agency                                                Agency   object      2946      0    0.000000      52
Posting Type                                    Posting Type   object      2946      0    0.000000       2
# Of Positions                                # Of Positions    int64      2946      0    0.000000      34
Business Title                                Business Title   object      2946      0    0.000000    1244
Civil Service Title                      Civil Service Title   object      2946      0    0.000000     312
Title Code No                                  Title Code No   object      2946      0    0.000000     323
Level                                                  Level   object      2946      0    0.000000      14
Job Category        

In [33]:
import plotly.express as px
import plotly.graph_objects as go

logger.info("Starting chart generation for exploratory data")

column_summary = column_summary.rename(columns={"null_pct": "Null Percentage", "unique": "Unique Values", "column": "Column"})


# chart 1: show which columns have the most missing values
null_plot = px.bar(
    column_summary.sort_values('Null Percentage', ascending=False),
    x='Column', y='Null Percentage',
    title='Columns Ranked by % Missing',
    labels={'Column': 'Field', 'Null Percentage': '% missing'},
    color='Null Percentage', color_continuous_scale='Reds'
)
null_plot.update_layout(xaxis_tickangle=-45, height=500)
null_plot.show()

# chart 2: highest cardinality columns in terms of unique values (top 15)
uniq_plot = px.bar(
    column_summary.sort_values('Unique Values', ascending=False).head(15),
    x='Column', y='Unique Values',
    title='Top 15 Fields by Unique Count',
    labels={'Column': 'Field', 'Unique Values': 'Count'},
    color='Unique Values', color_continuous_scale='Blues'
)
uniq_plot.update_layout(xaxis_tickangle=-45, height=500)
uniq_plot.show()

# chart 3: heatmap summarizing numeric statistics
stats_matrix = column_summary.describe().T
heat = go.Figure(data=go.Heatmap(
    z=stats_matrix.values,
    x=stats_matrix.columns,
    y=stats_matrix.index,
    colorscale='Viridis'
))
heat.update_layout(title='Numeric Summary Heatmap', height=500, width=900)
heat.show()

# chart 4: breakdown of categorical variables (limited cardinality)
logger.info("Preparing categorical frequency chart")
cat_rows = []
for field, freqdict in category_counts.items():
    for val, cnt in list(freqdict.items())[:10]:
        cat_rows.append({'Field': field, 'Value': str(val), 'Count': cnt})

cat_df = pd.DataFrame(cat_rows)
cat_chart = px.bar(
    cat_df,
    x='Value', y='Count', color='Field',
    title='Sample Values from Categorical Fields',
    labels={'Value': 'Category', 'Count': 'Occurrences'},
    facet_col='Field', facet_col_wrap=3
)
cat_chart.update_layout(height=600)
cat_chart.show()

logger.info("Charts for EDA rendered")

2026-02-27 20:46:42,605 - pre_process_functions - INFO - Starting chart generation for exploratory data


2026-02-27 20:46:43,214 - pre_process_functions - INFO - Preparing categorical frequency chart


2026-02-27 20:46:43,347 - pre_process_functions - INFO - Charts for EDA rendered


#### Detailed Findings

Complete insights and summary statistics have been documented in MyDocument.md

Reference: [MyDocument.md](MyDocument.md) <br><br>

Continuing transformation based on the insights gathered above

### Preprocessing Dataset

In [34]:
logger.info("Preparing preprocessing arguments")
#Preparing Arguments
rename_col_mapping_path = 'ColumnsMapping.json'
remove_special_chars_cols = ['primary_work_location','job_description','division_unit','min_qualify_requirements','preferred_skills',]
convert_to_double_cols = ['salary_min_range','salary_max_range']
convert_to_datetime_cols = ['job_posting_start_date','job_posting_updated_date']
convert_to_titlecase_cols = ['agency_name','business_title','civil_service_title','job_category','primary_work_location','division_unit']

columns_to_drop = ["recruitment_contact", "job_posting_end_date", "shift_schedule", "secondary_work_location", "application_instructions", "residency_requirement", "additional_info","process_date"]

remove_duplicates_params = [['job_id','posting_type'],['job_posting_updated_date'],True]
    
logger.info("Preprocessing arguments prepared successfully")

2026-02-27 20:46:43,375 - pre_process_functions - INFO - Preparing preprocessing arguments
2026-02-27 20:46:43,377 - pre_process_functions - INFO - Preprocessing arguments prepared successfully


In [35]:
logger.info("Applying preprocessing to DataFrame")
df = pre_process_data(df,
                      rename_col_mapping_path=rename_col_mapping_path,
                      remove_special_chars_cols=remove_special_chars_cols,
                      convert_to_double_cols=convert_to_double_cols,
                      convert_to_datetime_cols=convert_to_datetime_cols,
                      convert_to_titlecase_cols=convert_to_titlecase_cols,
                      remove_duplicates_params=remove_duplicates_params,
                      drop_columns_list = columns_to_drop
                      )
logger.info(f"Preprocessing completed successfully")


2026-02-27 20:46:43,442 - pre_process_functions - INFO - Applying preprocessing to DataFrame
2026-02-27 20:46:43,443 - pre_process_functions - INFO - Beginning preprocessing routine
2026-02-27 20:46:43,655 - pre_process_functions - INFO - Step 1/6: Applying column mapping from ColumnsMapping.json
2026-02-27 20:46:43,657 - root - INFO - Checking if the input is Pyspark DataFrame or not
2026-02-27 20:46:43,658 - root - INFO - Input is a Spark DataFrame. Proceeding with column renaming.
2026-02-27 20:46:43,659 - root - INFO - Loading column mapping from path : ColumnsMapping.json
2026-02-27 20:46:43,668 - root - INFO - Column mapping loaded successfully , Proceeding with column renaming
2026-02-27 20:46:43,745 - root - INFO - Columns rename completed as per mapping
2026-02-27 20:46:43,746 - pre_process_functions - INFO - Step 2/6: Dropping 8 columns
2026-02-27 20:46:43,746 - root - INFO - Dropping 8 columns: ['recruitment_contact', 'job_posting_end_date', 'shift_schedule', 'secondary_work

In [36]:
display(df)

+------+------------------------------+------------+------------------------+------------------------------------------------------------------------------------+------------------------------+----------+---------+--------------------------------------------------------------------------------------------------+--------------------+----------------+----------------+----------------+------------------------------+------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Deduplication & Clubbing Based on Job ID & Posting Type

In [37]:
logger.info("Starting deduplication and clubbing based on Job ID and Posting Type")
logger.info("Step 1: Creating window specifications for partitioning")

# Window spec 1: Partition by job_id to group all postings of the same job together
win_job_id = Window.partitionBy('job_id')
logger.info("Window 1 created: Partitioning by job_id")

# Window spec 2: Partition by job_id and order by posting date (descending) to identify most recent posting
win_job_post_update = Window.partitionBy('job_id').orderBy(col('job_posting_updated_date').desc())
logger.info("Window 2 created: Partitioning by job_id, ordering by job_posting_updated_date DESC")

# Count rows before deduplication
rows_before = df.count()
logger.info(f"Row count before deduplication: {rows_before}")

logger.info("Step 2: Applying transformation ")
# Transformation :
# 1. collect_set: Gather all unique posting_types for each job_id into an array
# 2. concat_ws + array_sort: Combine posting_types into a single string (sorted for consistency)
# 3. row_number: Assign rank to each job posting by recency (most recent = 1)
# 4. filter: Keep only the most recent posting for each job_id (rank = 1)
# 5. drop: Remove the temporary rank column
df = df\
        .withColumn('posting_type', collect_set('posting_type').over(win_job_id))\
        .withColumn('posting_type', concat_ws("/", array_sort(col('posting_type'))))\
        .withColumn('rnk', row_number().over(win_job_post_update))\
        .filter(col('rnk') == 1)\
        .drop('rnk')

logger.info("Transformation completed")

# Count rows after deduplication
rows_after = df.count()
logger.info(f"Row count after deduplication: {rows_after}")
logger.info(f"Deduplication summary: {rows_before} rows reduced to {rows_after} rows (removed {rows_before - rows_after} duplicates)")

logger.info("Displaying deduplicated DataFrame")
display(df)


2026-02-27 20:46:49,197 - pre_process_functions - INFO - Starting deduplication and clubbing based on Job ID and Posting Type
2026-02-27 20:46:49,208 - pre_process_functions - INFO - Step 1: Creating window specifications for partitioning
2026-02-27 20:46:49,268 - pre_process_functions - INFO - Window 1 created: Partitioning by job_id
2026-02-27 20:46:49,310 - pre_process_functions - INFO - Window 2 created: Partitioning by job_id, ordering by job_posting_updated_date DESC
2026-02-27 20:46:52,063 - pre_process_functions - INFO - Row count before deduplication: 2914
2026-02-27 20:46:52,066 - pre_process_functions - INFO - Step 2: Applying transformation 
2026-02-27 20:46:52,425 - pre_process_functions - INFO - Transformation completed
2026-02-27 20:46:56,994 - pre_process_functions - INFO - Row count after deduplication: 1661
2026-02-27 20:46:56,998 - pre_process_functions - INFO - Deduplication summary: 2914 rows reduced to 1661 rows (removed 1253 duplicates)
2026-02-27 20:46:57,003 - 

+------+------------------------------+-----------------+------------------------+------------------------------------------------------------------------------------------------------+------------------------------+----------+---------+---------------------------------------------------------------------------+--------------------+----------------+----------------+----------------+------------------------------+------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [38]:
logger.info("Checking for remaining duplicates on job_id")
# check duplicates on job_id
dup_check = df.groupBy(['job_id']).agg(count(lit(1)).alias('cnt')).filter('cnt > 1')
dup_count = dup_check.count()
logger.info(f"Duplicate check completed: Found {dup_count} remaining duplicates")
display(dup_check)


2026-02-27 20:47:01,605 - pre_process_functions - INFO - Checking for remaining duplicates on job_id
2026-02-27 20:47:06,633 - pre_process_functions - INFO - Duplicate check completed: Found 0 remaining duplicates


+------+---+
|job_id|cnt|
+------+---+
+------+---+



## Adding Annualzied salary columns & Adding  Qualification Indicator Column

In [39]:
logger.info("Adding annualized salary and qualification indicator columns")
df = annualize_salary(df)
df = create_qualification_indicator(df)
logger.info("New Annulaized and qualification indicator columns added successfully")
display(df)


2026-02-27 20:47:11,353 - pre_process_functions - INFO - Adding annualized salary and qualification indicator columns


2026-02-27 20:47:11,360 - root - INFO - Starting salary annualization
2026-02-27 20:47:11,789 - root - INFO - Salary annualization completed
2026-02-27 20:47:11,790 - root - INFO - Creating qualification indicator column
2026-02-27 20:47:11,843 - root - INFO - Qualification indicator column created
2026-02-27 20:47:11,859 - pre_process_functions - INFO - New Annulaized and qualification indicator columns added successfully


+------+------------------------------+-----------------+------------------------+------------------------------------------------------------------------------------------------------+------------------------------+----------+---------+---------------------------------------------------------------------------+--------------------+----------------+----------------+----------------+------------------------------+------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Whats the number of jobs posting per category (Top 10)?

In [40]:
logger.info("Retrieving top 10 job categories by posting frequency")
# Group by job category, count the number of job postings per category, and sort in descending order
job_category_stats = df.groupBy(['job_category']) \
    .agg(count(col('job_id')).alias('job_count')) \
    .orderBy(col('job_count').desc()) \
    .limit(10)
logger.info("Top 10 job categories analysis complete ")
logger.debug(f"Retrieved {job_category_stats.count()} top job categories")
display(job_category_stats)


2026-02-27 20:47:16,765 - pre_process_functions - INFO - Retrieving top 10 job categories by posting frequency
2026-02-27 20:47:16,856 - pre_process_functions - INFO - Top 10 job categories analysis complete 


+-----------------------------------------+---------+
|job_category                             |job_count|
+-----------------------------------------+---------+
|Engineering, Architecture, & Planning    |260      |
|Technology, Data & Innovation            |182      |
|Legal Affairs                            |120      |
|Building Operations & Maintenance        |99       |
|Finance, Accounting, & Procurement       |98       |
|Public Safety, Inspections, & Enforcement|98       |
|Administration & Human Resources         |88       |
|Health                                   |71       |
|Constituent Services & Community Programs|68       |
|Policy, Research & Analysis              |64       |
+-----------------------------------------+---------+



# Whats the salary distribution per job category?

In [41]:
logger.info("Analyzing salary distribution per job category")
salary_by_category = df.groupBy(['job_category']).agg(
            round(avg(col('annualized_salary_avg_range')),2).alias('avg_salary'),
            round(avg(col('annualized_salary_min_range')),2).alias('avg_min_salary'),
            round(avg(col('annualized_salary_max_range')),2).alias('avg_max_salary')
    ).orderBy(col('avg_salary').desc())
logger.info("Salary distribution analysis completed")
display(salary_by_category)


2026-02-27 20:47:29,309 - pre_process_functions - INFO - Analyzing salary distribution per job category
2026-02-27 20:47:29,389 - pre_process_functions - INFO - Salary distribution analysis completed


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+--------------+--------------+
|job_category                                                                                                                                                                                             |avg_salary|avg_min_salary|avg_max_salary|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+--------------+--------------+
|Administration & Human Resources Finance, Accounting, & Procurement Building Operations & Maintenance                                                                                                    |218587.0  |218587.0      |218587.0      |
|Engineering, Archit

# Is there any correlation between the higher degree and the salary?

In [42]:
logger.info("Evaluating statistical relationship between degree requirement and salary")
logger.debug("Computing correlation between is_degree_req and annualized_salary_avg_range")

# Calculate correlation between:
# - is_degree_req: Binary indicator if degree is required (1 = yes, 0 = no)
# - annualized_salary_avg_range: Average annualized salary for the position
correlation = df.stat.corr("is_degree_req", "annualized_salary_avg_range")

logger.info(f"Correlation coefficient: {correlation:.4f}")
logger.debug("Analysis: A positive value indicates positions requiring degrees tend to have higher salaries &" \
" negative value indicates the opposite")

print(f"Correlation between degree requirement and salary: {correlation:.4f}")


2026-02-27 20:47:35,963 - pre_process_functions - INFO - Evaluating statistical relationship between degree requirement and salary
2026-02-27 20:47:40,921 - pre_process_functions - INFO - Correlation coefficient: 0.0490


Correlation between degree requirement and salary: 0.0490


In [43]:
logger.info("Analyzing salary by degree requirement")
salary_by_degree = df.groupBy(['is_degree_req']).agg(
            round(avg(col('annualized_salary_min_range')),2).alias('avg_min_salary'),
            round(avg(col('annualized_salary_max_range')),2).alias('avg_max_salary'),
            round(avg(col('annualized_salary_avg_range')),2).alias('avg_salary'),
    ).orderBy(col('avg_min_salary').desc(), col('avg_max_salary').desc())
logger.info("Salary by degree requirement analysis completed")
display(salary_by_degree)

2026-02-27 20:47:40,980 - pre_process_functions - INFO - Analyzing salary by degree requirement


2026-02-27 20:47:41,140 - pre_process_functions - INFO - Salary by degree requirement analysis completed


+-------------+--------------+--------------+----------+
|is_degree_req|avg_min_salary|avg_max_salary|avg_salary|
+-------------+--------------+--------------+----------+
|1            |64311.24      |91448.67      |77880.05  |
|0            |61097.36      |88545.46      |74821.78  |
+-------------+--------------+--------------+----------+



# Whats the job posting having the highest salary per agency?

In [44]:
logger.info("Identifying highest-compensated job posting within each agency")
logger.debug("Creating window specification for ranking by agency and salary")

from pyspark.sql.window import Window

# Define window specificationm partiton by agency_name & order by annualized_salary_avg_range descending
WinSpec = Window.partitionBy(col('agency_name')).orderBy(col('annualized_salary_avg_range').desc())

logger.debug("Window specification created: partitioned by agency_name, ordered by salary descending")

# Assign ranking: dense_rank() assigns sequential ranks within each partition
# Highest salary in each agency gets rank = 1
df = df.withColumn('job_posting_rank', dense_rank().over(WinSpec))

logger.debug("Dense rank column added to DataFrame")

# Filter for top-ranked position (rank = 1) per agency
# Select relevant salary and position details
highest_paid_jobs = df.filter(col('job_posting_rank') == 1) \
    .select([
        'agency_name',
        'business_title',
        'annualized_salary_avg_range',
        'annualized_salary_min_range',
        'annualized_salary_max_range'
    ])

logger.info("Highest paid job posting identified per agency")
logger.debug(f"Retrieved {highest_paid_jobs.count()} highest-paying positions (one per agency)")
display(highest_paid_jobs)


2026-02-27 20:47:47,540 - pre_process_functions - INFO - Identifying highest-compensated job posting within each agency


2026-02-27 20:47:47,818 - pre_process_functions - INFO - Highest paid job posting identified per agency


+------------------------------+--------------------------------------------------------------------------------+---------------------------+---------------------------+---------------------------+
|agency_name                   |business_title                                                                  |annualized_salary_avg_range|annualized_salary_min_range|annualized_salary_max_range|
+------------------------------+--------------------------------------------------------------------------------+---------------------------+---------------------------+---------------------------+
|Fire Department               |Senior Enterprise Applications Integration Developer                            |120474.5                   |96020.0                    |144929.0                   |
|Law Department                |Deputy Chief Of Information Technology                                          |164104.0                   |164104.0                   |164104.0                   |
|Nyc Housi

### Whats the job positings average salary per agency for the last 2 years?

In [45]:
logger.info("Finding maximum job posting date")
df_max = df.withColumn('job_posting_updated_date',to_timestamp(col('job_posting_updated_date')))
max_date_result = df_max.agg(max(col('job_posting_updated_date'))).collect()

max_date = max_date_result[0][0] # Max date found to 2020-06-08
print(f"Maximum job posting date: {max_date}")
logger.info(f"Maximum job posting date: {max_date}")

2026-02-27 20:48:00,969 - pre_process_functions - INFO - Finding maximum job posting date
2026-02-27 20:48:05,787 - pre_process_functions - INFO - Maximum job posting date: 2020-06-08 00:00:00


Maximum job posting date: 2020-06-08 00:00:00


In [46]:
logger.info("Analyzing job postings and average salary for last 2 years")
# Assuming the current date as 2020-04-01 00:00:00 to get some tangable results , in case of real time we can use current_timestamp() function instead of hardcoding the date
df_filter = df.filter("job_posting_updated_date >= ADD_MONTHS(CAST('2020-04-01' AS DATE), -24)")
logger.info(f"Filtered to {df_filter.count()} job postings in last 2 years")

result = df_filter.groupBy(['agency_name','business_title']).agg(
    round(avg("annualized_salary_min_range"),2).alias("avg_salary_min"),
    round(avg("annualized_salary_max_range"),2).alias("avg_salary_max"),
    round(avg("annualized_salary_avg_range"),2).alias('avg_salary'))

logger.info("2-year analysis completed")
display(result)

2026-02-27 20:48:05,922 - pre_process_functions - INFO - Analyzing job postings and average salary for last 2 years
2026-02-27 20:48:10,435 - pre_process_functions - INFO - Filtered to 853 job postings in last 2 years
2026-02-27 20:48:10,507 - pre_process_functions - INFO - 2-year analysis completed


+------------------------------+-------------------------------------------------------------------+--------------+--------------+----------+
|agency_name                   |business_title                                                     |avg_salary_min|avg_salary_max|avg_salary|
+------------------------------+-------------------------------------------------------------------+--------------+--------------+----------+
|Dept Of Parks & Recreation    |Deputy Director Of Gis & Analytics                                 |75000.0       |85000.0       |80000.0   |
|Mayors Office Of Contract Svcs|Technical Senor Business Analyst, Technology Innovation & Solutions|90000.0       |100000.0      |95000.0   |
|Department Of Buildings       |Supervising Inspector, Elevators                                   |61010.0       |70161.0       |65585.5   |
|Admin For Children's Svcs     |Community Assessment Specialist-queens                             |70900.0       |84547.0       |77723.5   |
|Finan

# What are the highest paid skills in the US market?

In [47]:
logger.info("Starting analysis of highest paid skills in job market")
logger.info("Step 1: Extracting and normalizing individual skills from preferred_skills column")

# Transform 1: Convert preferred_skills text to lowercase for case-insensitive processing
df_skills = df.withColumn("preferred_skills", lower(col("preferred_skills")))
logger.debug("Step 1a: Converted preferred_skills to lowercase")

# Transform 2: Remove special characters, keeping only letters and whitespace
df_skills = df_skills.withColumn("preferred_skills", regexp_replace(col("preferred_skills"), "[^a-zA-Z\\s]", ""))
logger.debug("Step 1b: Removed special characters from preferred_skills")

# Transform 3: Split skills by whitespace and explode into individual rows
# This creates one row per skill, enabling per-skill analysis
df_skills = df_skills.withColumn("skill", explode(split(col("preferred_skills"), "\\s+")))
logger.info("Step 1 complete: Skills extracted into individual rows")

logger.info("Step 2: Filtering out common stop words")
# Define stop words that don't represent actual job skills
stop_words = ["the", "and", "to", "of", "in", "a", "with", "for", "or", "skills", "experience", 
              "ability", "knowledge", "", "strong", "excellent", "must", "be", "is"]
logger.debug(f"Defined {len(stop_words)} stop words for filtering")

# Remove rows containing stop words
df_filtered_skills = df_skills.filter(~col("skill").isin(stop_words))
initial_skill_count = df_skills.count()
filtered_skill_count = df_filtered_skills.count()
logger.info(f"Stop word filtering complete: {initial_skill_count} total skills -> {filtered_skill_count} meaningful skills")

logger.info("Step 3: Aggregating skills by compensation metrics")
logger.debug("Grouping by skill and calculating salary statistics")

top_paid_skills = df_filtered_skills.groupBy("skill") \
    .agg(
        avg("annualized_salary_avg_range").alias("avg_skill_salary"),
        round(avg("annualized_salary_min_range"), 2).alias("avg_skill_salary_min"),
        round(avg("annualized_salary_max_range"), 2).alias("avg_skill_salary_max"),
        count("job_id").alias("frequency")
    ) \
    .filter(col("frequency") > 10) \
    .orderBy(desc("avg_skill_salary")) \
    .limit(20)

logger.info("Aggregation and filtering complete")
logger.debug("Applied filters: minimum frequency > 10 occurrences, selected top 20 by average salary")

result_count = top_paid_skills.count()
logger.info(f"Top {result_count} highest-paid skills identified (with frequency > 10)")
logger.info("Skills analysis complete - displaying results")
display(top_paid_skills)


2026-02-27 20:48:15,508 - pre_process_functions - INFO - Starting analysis of highest paid skills in job market
2026-02-27 20:48:15,510 - pre_process_functions - INFO - Step 1: Extracting and normalizing individual skills from preferred_skills column
2026-02-27 20:48:15,646 - pre_process_functions - INFO - Step 1 complete: Skills extracted into individual rows
2026-02-27 20:48:15,648 - pre_process_functions - INFO - Step 2: Filtering out common stop words
2026-02-27 20:48:23,079 - pre_process_functions - INFO - Stop word filtering complete: 65121 total skills -> 43070 meaningful skills
2026-02-27 20:48:23,081 - pre_process_functions - INFO - Step 3: Aggregating skills by compensation metrics
2026-02-27 20:48:23,191 - pre_process_functions - INFO - Aggregation and filtering complete
2026-02-27 20:48:35,236 - pre_process_functions - INFO - Top 20 highest-paid skills identified (with frequency > 10)
2026-02-27 20:48:35,238 - pre_process_functions - INFO - Skills analysis complete - displa

+----------------+------------------+--------------------+--------------------+---------+
|skill           |avg_skill_salary  |avg_skill_salary_min|avg_skill_salary_max|frequency|
+----------------+------------------+--------------------+--------------------+---------+
|investment      |125313.63636363637|118000.0            |132627.27           |11       |
|wastewater      |109307.09         |83677.54            |134936.64           |20       |
|teams           |108120.28571428571|79362.59            |136877.99           |28       |
|resolve         |107910.59999999999|95808.03            |120011.43           |12       |
|requires        |107814.11538461539|91795.46            |123832.77           |13       |
|vulnerability   |107272.72727272728|75000.0             |139545.45           |11       |
|aws             |106844.2          |77100.0             |136588.4            |15       |
|delivery        |106658.0          |85758.8             |127557.2            |15       |
|estate   

### Exporting the Dataset

In [48]:
logger.info("Starting dataset export")

export_to_csv(df,output_path="/dataset/cleaned",file_name="nyc-jobs-cleaned.csv")
logger.info("Dataset export completed successfully")

2026-02-27 20:48:44,407 - pre_process_functions - INFO - Starting dataset export
2026-02-27 20:48:44,412 - root - INFO - Starting CSV export: /dataset/cleaned/nyc-jobs-cleaned.csv
2026-02-27 20:48:49,085 - root - INFO - Number of rows to export: 1661
2026-02-27 20:48:58,816 - root - INFO - CSV export completed: /dataset/cleaned/nyc-jobs-cleaned.csv
2026-02-27 20:48:58,822 - pre_process_functions - INFO - Dataset export completed successfully
